In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import os
from astropy.utils.data import get_pkg_data_filename
from reproject import reproject_interp
from astropy import units as u
from reproject import reproject_from_healpix
from astropy_healpix import HEALPix
from astropy.coordinates import Galactic, ICRS
from astropy.coordinates import SkyCoord
from scipy.stats import linregress
import pandas as pd
import copy
from astropy.convolution import convolve
from astropy.convolution import Gaussian2DKernel
from reproject import reproject_to_healpix
import healpy as hp
from matplotlib import cm

## Becky's beam convolution code

In [ ]:
def convolve_to_beam(mapin, biggest_beam, current_beam,hdr, output = False):
    
    fwhm = np.sqrt(biggest_beam**2 - current_beam**2) * 60

    if output: print('Convolving', np.round(current_beam,2), 'deg to', np.round(biggest_beam,2), 'deg\t FWHM of kernel: '+str(fwhm/60)+' degrees')   
    pixsize = hdr['CDELT2']*60
    
    if fwhm < pixsize:
        if output: print("Just returning the map without change")
        return mapin
    
    if output: print('pixel size: '+str(pixsize)+' arcmin')   
    stddev = np.round((fwhm/2.355)/pixsize,0)
    if output: print('standard deviation of kernel: '+str(stddev)+' pixels')
    
    if stddev == 0:
        return mapin

    kernel = Gaussian2DKernel(x_stddev=stddev,y_stddev=stddev)
    
    return convolve(mapin,kernel)

## Following functions are from chime_dragons_decslices_v2

In [ ]:
def make_freq_list(dragons_f_file, freq_range = [500e6,730e6]):

    freq_dragons_all = np.genfromtxt(dragons_f_file)
    
    idx_keep = np.where((freq_dragons_all>freq_range[0]) & (freq_dragons_all<freq_range[1]))
    freq_list = freq_dragons_all[idx_keep]

    print('number of DRAGONS frequencies: ',len(freq_dragons_all))
    print('number of frequencies in selected range: ',len(freq_list))
    
    return freq_list

In [ ]:
def read_data(chime_files ,dragons_files, dragons_f_file, freq_list, grid_to='dragons'):

    print('')
    print('Reading in data...')
    print('')
    chime   = {'q':[],'u':[],'i':[]}
    dragons = {'q':[],'u':[],'i':[]}
        
    for stokes in ['q','u','i']:

        print('-------------------------------------')
        print('Stokes ',stokes)
        print('-------------------------------------')
        chime_stokes   = [] 
        dragons_stokes = []
        
        # read in data
        hdu_chime   = fits.open(chime_files[stokes])
        hdu_dragons = fits.open(dragons_files[stokes])
    
        # get headers
        hdr_chime   = hdu_chime[0].header
        hdr_dragons = hdu_dragons[0].header

        if stokes == 'q':
            # get frequencies
            freq_chime   = WCS(hdr_chime).all_pix2world(0,0,range(hdu_chime[0].data.shape[0]),0)[2]
            freq_dragons = np.genfromtxt(dragons_f_file)

        for i in range(0,len(freq_list)):
     
            # get closest DRAGONS frequency
            fidx_dragons = np.abs(freq_dragons-freq_list[i]).argmin()
            freq_pick    = freq_dragons[fidx_dragons]
            print('DRAGONS frequency: ',freq_dragons[fidx_dragons]/1e6,'MHz')
        
            # get closest CHIME frequency
            fidx_chime  = np.abs(freq_chime-freq_pick).argmin()
            print('CHIME frequency: ',freq_chime[fidx_chime]/1e6,'MHz')
    
            chime_stokes_slice   = hdu_chime[0].data[fidx_chime]
            dragons_stokes_slice = hdu_dragons[0].data[fidx_dragons]
    
            if grid_to == 'dragons':
                wcs     = WCS(hdr_dragons).dropaxis(2)
                chime_stokes_slice, footprint = reproject_interp((chime_stokes_slice,WCS(hdr_chime).dropaxis(2)),wcs)
            if grid_to == 'chime':
                wcs     = WCS(hdr_chime).dropaxis(2)
                dragons_stokes_slice, footprint = reproject_interp((dragons_stokes_slice,WCS(hdr_dragons).dropaxis(2)),wcs)

            chime_stokes.append(chime_stokes_slice)
            dragons_stokes.append(dragons_to_K(dragons_stokes_slice,freq_pick))

        chime[stokes]   = np.array(chime_stokes)
        dragons[stokes] = np.array(dragons_stokes)

        print('')
    
    return chime, dragons, wcs

In [ ]:
def dragons_to_K(dragons_Jy_slice,freq_use):

    JypK = np.load('/srv/data/dva/JypK.npy')
    allfreqs = np.load('/srv/data/dva/abscalfreqs.npy')
    #beam_eff = np.load('')
    beam_eff = 0.7

    fidx = np.abs(allfreqs-freq_use/1e6).argmin()

    print('Jy to K frequency: ',allfreqs[fidx],'MHz')
    print('DRAGONS conversion: ',JypK[fidx],'Jy per K')

    dragons_K_slice = (dragons_Jy_slice/JypK[fidx])/beam_eff
    print(dragons_K_slice.shape)
  
    return dragons_K_slice

In [ ]:
def mask_bright_srcs(chime, dragons, wcs, ddec=0.5):

    print('')
    print('Masking bright sources...')
    print('')

    cas_coord = SkyCoord('23h23m24.0s', '+58d48m54.0s', frame='icrs')
    cyg_coord = SkyCoord('19h59m28.4s', '+40d44m02.1s', frame='icrs')
    tau_coord = SkyCoord('5h34m31.9s', '+22d00m52.2s', frame='icrs')

    src_coords = [cas_coord,cyg_coord,tau_coord]

    decs_all = wcs.all_pix2world(0,range(dragons['q'].shape[1]),0)[1]
    RAs_all = wcs.all_pix2world(range(dragons['q'].shape[2]),0,0)[0]

    rara,decdec = np.meshgrid(RAs_all,decs_all)
    print(decdec.shape)
    print(rara.shape)
    
    for coord in src_coords:
        print(coord)
        rRA  = (rara - coord.ra.deg)*np.cos(decdec*np.pi/180)
        rdec = decdec - coord.dec.deg
        r2D = np.sqrt(rdec**2+rRA**2)

        for stokes in ['i','q','u']:
            print('Stokes ',stokes)
            for ifrq in range(0,chime[stokes].shape[0]):
                chime[stokes][ifrq][r2D < ddec/2] = np.nan
                dragons[stokes][ifrq][r2D < ddec/2] = np.nan

    return chime, dragons

In [ ]:
def mask_bright_regions(chime, dragons):

    print('')
    print('Masking bright regions...')
    print('')

    ## *** For now using first frequency channel for brightness thresholding
    wbright_pi = np.where( np.sqrt(chime['q'][0]**2+chime['u'][0]**2) > 5)
    wbright_i = np.where( chime['i'][0] > 30)

    for stokes in ['i','q','u']:
        print('Stokes ',stokes)
        for ifrq in range(0,chime[stokes].shape[0]):

            chime[stokes][ifrq][wbright_pi] = np.nan
            dragons[stokes][ifrq][wbright_pi] = np.nan
            chime[stokes][ifrq][wbright_i] = np.nan
            dragons[stokes][ifrq][wbright_i] = np.nan

    return chime, dragons

In [ ]:
def mask_bad_RA(chime, dragons, wcs):

    print('')
    print('Masking bad RA region...')
    print('')

    decs_all = wcs.all_pix2world(0,range(dragons['q'].shape[1]),0)[1]
    RAs_all = wcs.all_pix2world(range(dragons['q'].shape[2]),0,0)[0]

    wbad = np.where((RAs_all*24/360 > 17) & (RAs_all*24/360<21))

    for stokes in ['i','q','u']:
        print('Stokes ',stokes)
        for ifrq in range(0,chime[stokes].shape[0]):
            chime[stokes][ifrq][:,wbad] = np.nan
            dragons[stokes][ifrq][:,wbad] = np.nan

    return chime, dragons

In [ ]:
def make_all_maps(chime,dragons,wcs,freq_list,saveplots=False,outdir=None):

    print('')
    print('Making Stokes I, Q, U maps...')
    print('')
    
    for j in range(0,len(freq_list)):
        fig = plt.figure(figsize=(30,12))
        fs = 16
        
        crange = SkyCoord([24,0], [12,86], frame=ICRS, unit=(u.hr, u.deg))
        plt.subplots_adjust(hspace=0.01,wspace=0.01)
        
        axic = fig.add_subplot(321, projection=wcs.celestial)
        axqc = fig.add_subplot(323, projection=wcs.celestial)
        axuc = fig.add_subplot(325, projection=wcs.celestial)
        axid = fig.add_subplot(322, projection=wcs.celestial)
        axqd = fig.add_subplot(324, projection=wcs.celestial)
        axud = fig.add_subplot(326, projection=wcs.celestial)
        
        imic = axic.imshow(chime['i'][j],origin='lower',vmin=0,vmax=15,cmap='cubehelix')
        imqc = axqc.imshow(chime['q'][j],origin='lower',vmin=-1,vmax=1,cmap='RdBu_r')
        imuc = axuc.imshow(chime['u'][j],origin='lower',vmin=-1,vmax=1,cmap='RdBu_r')
        
        imid = axid.imshow(dragons['i'][j],origin='lower',vmin=0,vmax=15,cmap='cubehelix')
        imqd = axqd.imshow(dragons['q'][j],origin='lower',vmin=-1,vmax=1,cmap='RdBu_r')
        imud = axud.imshow(dragons['u'][j],origin='lower',vmin=-1,vmax=1,cmap='RdBu_r')
        
        axs = [axic,axqc,axuc,axid,axqd,axud]
        ims = [imic,imqc,imuc,imid,imqd,imud]
        labels = ['I (Jy/beam)','Q (Jy/beam)','U (Jy/beam)','I (K)','Q (K)','U (K)']
        
        for i in range(0,len(axs)):
            axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
            axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
            cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.65, pad=0.005, aspect=12)
            cbar.set_label(label=labels[i],fontsize=fs)
            cbar.ax.tick_params(labelsize=fs) 
            axs[i].set_xlabel(' ')
            axs[i].set_ylabel(' ')
            axs[i].tick_params(axis='both',labelsize=fs)
        axic.set_ylabel('dec',fontsize=fs)
        axqc.set_ylabel('dec',fontsize=fs)
        axuc.set_ylabel('dec',fontsize=fs)
        axuc.set_xlabel('RA',fontsize=fs)
        axud.set_xlabel('RA',fontsize=fs)

        plt.suptitle(str(np.round(freq_list[j]/1e6,3))+' MHz',fontsize=fs+4)

        if saveplots:
            out_file_name = 'dragons_chime_maps_'+str(np.round(freq_list[j]/1e6,3))+'_MHz.pdf'
            plt.savefig(outdir+out_file_name)
            plt.close(fig)

    return

In [ ]:
def dec_chime_vs_dragons(chime, dragons, wcs, freq_list, 
                         dec_range=[15,85], ddec=1, make_plots = False,outdir=None):
    
    print('')
    print('Calculating slopes and intercepts...')
    print('')
    
    stokes = ['i','q','u']

    decs = wcs.all_pix2world(0,range(dragons['q'].shape[1]),0)[1]
    decs_sampled = decs[800:1440][::8]

    print(decs)
    slopes = {'i':np.empty([len(freq_list),len(decs)]),
              'q':np.empty([len(freq_list),len(decs)]),
              'u':np.empty([len(freq_list),len(decs)])}
    ints   = {'i':np.empty([len(freq_list),len(decs)]),
              'q':np.empty([len(freq_list),len(decs)]),
              'u':np.empty([len(freq_list),len(decs)])}

    for i in range(0,len(decs)):
        print(decs[i])
        idx = np.where((decs <= decs[i]+ddec/2) & (decs >= decs[i]-ddec/2))
        
        for j in range(0,len(freq_list)):
            #print(freq_list[j])

            if (make_plots == True) & (decs[i] in decs_sampled):
                print(decs[i])
                fig,ax = plt.subplots(1,3,figsize=(15,5))
    
            for ii in range(0,3):
                try:
                    wgood = np.where( np.isfinite(chime[stokes[ii]][j,idx]) & np.isfinite(dragons[stokes[ii]][j,idx]) )  
                    x = chime[stokes[ii]][j,idx][wgood].flatten()
                    y = dragons[stokes[ii]][j,idx][wgood].flatten()
                    fit = linregress(x,y)
                    slopes[stokes[ii]][j,i] = round(fit.slope,3)
                    ints[stokes[ii]][j,i] = round(fit.intercept,3) 
        
                    if (make_plots == True) & (decs[i] in decs_sampled):
                        ax[ii].scatter(x,y,s=2,color='C'+str(ii))
                        ax[ii].plot(np.linspace(-30,30),fit.slope*np.linspace(-30,30)+fit.intercept,color='k')
                        if ii > 0:
                            ax[ii].set_xlim(-2,2)
                            ax[ii].set_ylim(-2,2)
                        else:
                            ax[ii].set_xlim(-15,15)
                            ax[ii].set_ylim(-15,15)
                        ax[ii].set_title('Stokes '+stokes[ii].upper()+', '+str(np.round(freq_list[j]/1e6,3))+' MHz, dec='+str(decs[i]))
                        ax[ii].grid()
                        ax[ii].set_aspect('equal')
                        ax[ii].set_xlabel('CHIME (Jy/beam)')
                        ax[ii].set_ylabel('DRAGONS (K)')
                except:
                    pass

            if (make_plots == True) & (decs[i] in decs_sampled):
                out_file_name = str(np.round(freq_list[j]/1e6,3))+'_MHz_dec'+str(np.round(decs[i],3))+'.png'
                print(out_file_name)
                plt.savefig(outdir+'dragons_versus_chime_'+out_file_name)
                plt.close(fig)
        
    return decs, slopes, ints

In [ ]:
def zoom_plots(crange,vmax=1,stokes='q'):

    fig = plt.figure(figsize=(20,6))
    fs = 16
        
    plt.subplots_adjust(hspace=0.0)
        
    ax_chime   = fig.add_subplot(131, projection=wcs.celestial)
    ax_dragons = fig.add_subplot(132, projection=wcs.celestial)
    ax_combine = fig.add_subplot(133, projection=wcs.celestial)
    
    freq_title = str(np.round(freq_list[fidx]/1e6,3))+' MHz'
    
    ax_chime.set_title('CHIME '+freq_title,fontsize=fs)
    ax_dragons.set_title('DRAGONS '+freq_title,fontsize=fs)
    ax_combine.set_title('combined '+freq_title,fontsize=fs)
    
    im_chime   = ax_chime.imshow(chime_K[stokes][0],vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im_dragons = ax_dragons.imshow(dragons[stokes][0],vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im_combine = ax_combine.imshow(combined[stokes][0],vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    
    axs = [ax_chime,ax_dragons,ax_combine]
    ims = [im_chime,im_dragons,im_combine]
        
    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        #cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.65, pad=0.005, aspect=12)
        #cbar.set_label(label='K',fontsize=fs)
        #cbar.ax.tick_params(labelsize=fs) 
        axs[i].set_xlabel('RA',fontsize=fs)
        axs[i].set_ylabel(' ')
        axs[i].tick_params(axis='both',labelsize=fs)
    ax_chime.set_ylabel('dec',fontsize=fs)

    return

## Read in CHIME, DRAGONS data, regrid to CHIME, convert DRAGONS to K

In [ ]:
freq_list = [612e6]

dragons_f_file = '/srv/data/dragons/Mar2024_500_1000_MHz_cube/ListOfFrequencies_Hz.txt'

chime_files   = {'q':'/srv/data/chime/chime_QU_Oct2023_400_729/Q_400_729_Oct2023_new_eq.fits',
                     'u':'/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2024_new.fits',
                     'i':'/srv/data/chime/chime_QU_Oct2023_400_729/I_400_729_Oct2023_new.fits'}
    
dragons_files = {'q':'/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_q_cube.fits',
                 'u':'/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_u_cube.fits',
                 'i':'/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_i_cube.fits'}

chime, dragons, wcs = read_data(chime_files ,dragons_files, dragons_f_file, freq_list, grid_to='chime')

## Get Jy/beam -> K conversion parameters from DRAGONS

In [ ]:
chime, dragons = mask_bright_srcs(chime, dragons, wcs, ddec=6)
chime, dragons = mask_bright_regions(chime, dragons)
chime, dragons = mask_bad_RA(chime, dragons, wcs)

make_all_maps(chime,dragons,wcs,freq_list,saveplots=False)

decs, slopes, ints = dec_chime_vs_dragons(chime, dragons, wcs, freq_list, dec_range=[15,85], ddec=2, make_plots=False)


In [ ]:
fidx = 0

## Apply conversion to K for CHIME

In [ ]:
print(slopes['q'].shape)

slopes_arr = {'q':[],'u':[],'i':[]}
ints_arr   = {'q':[],'u':[],'i':[]}

for stokes in ['q','u','i']:
    slopes_arr[stokes] = np.expand_dims(slopes[stokes],axis=2)
    ints_arr[stokes]   = np.expand_dims(ints[stokes],axis=2)

print(slopes_arr['q'].shape)
print(chime['q'].shape)

chime, dragons, wcs = read_data(chime_files ,dragons_files, dragons_f_file, freq_list, grid_to='chime')

chime_K = {'i':np.empty_like(chime['i']),'q':np.empty_like(chime['q']),'u':np.empty_like(chime['u'])}
for stokes in ['q','u','i']:
    chime_K[stokes][fidx] = chime[stokes][fidx]*slopes_arr[stokes][fidx] + ints_arr[stokes][fidx]

## Smooth CHIME to DRAGONS resolution

In [ ]:
chime_beam   = 25/60
dragons_beam = 2.

print('beam ratio: ',dragons_beam/chime_beam)

chime_K_smooth = {'i':np.empty_like(chime['i']),'q':np.empty_like(chime['q']),'u':np.empty_like(chime['u'])}

for stokes in ['q','u','i']:
    print('Convolving Stokes ',stokes.upper())
    chime_K_smooth[stokes][fidx] = convolve_to_beam(chime_K[stokes][fidx], dragons_beam, chime_beam, wcs.to_header(), output = True)
    print('')

## Calculate CHIME residuals (DRAGONS - smoothed CHIME)

In [ ]:
chime_K_res = {'i':np.empty_like(chime['i']),'q':np.empty_like(chime['q']),'u':np.empty_like(chime['u'])}

for stokes in ['q','u','i']:
    print('Residuals for ',stokes.upper())
    chime_K_res[stokes] = dragons[stokes] - chime_K_smooth[stokes]
    

## Combine DRAGONS and CHIME (CHIME + residuals)

In [ ]:
combined = {'i':np.empty_like(chime['i']),'q':np.empty_like(chime['q']),'u':np.empty_like(chime['u'])}

for stokes in ['q','u','i']:
    print('Residuals for ',stokes.upper())
    combined[stokes] = chime_K[stokes] + chime_K_res[stokes]
    

## Make all the maps

In [ ]:
fig = plt.figure(figsize=(20,22))
fs = 16
stokes = 'u'
    
crange = SkyCoord([24,0], [12,86], frame=ICRS, unit=(u.hr, u.deg))
plt.subplots_adjust(hspace=0.0)
    
ax_chime   = fig.add_subplot(511, projection=wcs.celestial)
ax_dragons = fig.add_subplot(512, projection=wcs.celestial)
ax_chimesm = fig.add_subplot(513, projection=wcs.celestial)
ax_resid   = fig.add_subplot(514, projection=wcs.celestial)
ax_combine = fig.add_subplot(515, projection=wcs.celestial)

freq_title = str(np.round(freq_list[fidx]/1e6,3))+' MHz'

ax_chime.set_title('CHIME (converted to K with DRAGONS) '+freq_title,fontsize=fs)
ax_dragons.set_title('DRAGONS '+freq_title,fontsize=fs)
ax_chimesm.set_title('CHIME smoothed to DRAGONS resolution '+freq_title,fontsize=fs)
ax_resid.set_title('smoothed CHIME residuals '+freq_title,fontsize=fs)
ax_combine.set_title('combined (CHIME+residuals) '+freq_title,fontsize=fs)

im_chime   = ax_chime.imshow(chime_K[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')
im_dragons = ax_dragons.imshow(dragons[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')
im_chimesm = ax_chimesm.imshow(chime_K_smooth[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')
im_resid   = ax_resid.imshow(chime_K_res[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')
im_combine = ax_combine.imshow(combined[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')

axs = [ax_chime,ax_dragons,ax_chimesm,ax_resid,ax_combine]
ims = [im_chime,im_dragons,im_chimesm,im_resid,im_combine]
    
for i in range(0,len(axs)):
    axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
    axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
    cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.65, pad=0.005, aspect=12)
    cbar.set_label(label='K',fontsize=fs)
    cbar.ax.tick_params(labelsize=fs) 
    axs[i].set_xlabel(' ')
    axs[i].set_ylabel('dec',fontsize=fs)
    axs[i].tick_params(axis='both',labelsize=fs)
ax_combine.set_xlabel('RA',fontsize=fs)

plt.savefig('../PLOTS/chime_dragons_combine_procedure_u.png')

## Compare CHIME, DRAGONS, combined, full sky

In [ ]:
fig = plt.figure(figsize=(20,13))
fs = 16
stokes = 'q'
    
crange = SkyCoord([24,0], [12,86], frame=ICRS, unit=(u.hr, u.deg))
plt.subplots_adjust(hspace=0.0)
    
ax_chime   = fig.add_subplot(311, projection=wcs.celestial)
ax_dragons = fig.add_subplot(312, projection=wcs.celestial)
ax_combine = fig.add_subplot(313, projection=wcs.celestial)

freq_title = str(np.round(freq_list[fidx]/1e6,3))+' MHz'

ax_chime.set_title('CHIME (converted to K with DRAGONS) '+freq_title,fontsize=fs)
ax_dragons.set_title('DRAGONS '+freq_title,fontsize=fs)
ax_combine.set_title('combined (CHIME+residuals) '+freq_title,fontsize=fs)

im_chime   = ax_chime.imshow(chime_K[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')
im_dragons = ax_dragons.imshow(dragons[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')
im_combine = ax_combine.imshow(combined[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')

axs = [ax_chime,ax_dragons,ax_combine]
ims = [im_chime,im_dragons,im_combine]
    
for i in range(0,len(axs)):
    axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
    axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
    cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.65, pad=0.005, aspect=12)
    cbar.set_label(label='K',fontsize=fs)
    cbar.ax.tick_params(labelsize=fs) 
    axs[i].set_xlabel(' ')
    axs[i].set_ylabel('dec',fontsize=fs)
    axs[i].tick_params(axis='both',labelsize=fs)
ax_combine.set_xlabel('RA',fontsize=fs)

plt.savefig('../PLOTS/chime_dragons_combined_q.png')

## Fan region zoom

In [ ]:
crange = SkyCoord([7,1], [30,80], frame=ICRS, unit=(u.hr, u.deg))
zoom_plots(crange,vmax=2,stokes='q')

plt.savefig('../PLOTS/chime_dragons_combined_q_Fan.png')

## Tadpole zoom

In [ ]:
crange = SkyCoord([4,2], [50,80], frame=ICRS, unit=(u.hr, u.deg))
zoom_plots(crange,vmax=2.5,stokes='u')

## Per-Tau shell zoom

In [ ]:
crange = SkyCoord([6,2], [12,60], frame=ICRS, unit=(u.hr, u.deg))
zoom_plots(crange,vmax=1,stokes='u')

plt.savefig('../PLOTS/chime_dragons_combined_u_PerTau.png')

## NPS zoom

In [ ]:
crange = SkyCoord([16,12], [12,60], frame=ICRS, unit=(u.hr, u.deg))
zoom_plots(crange,vmax=1.5,stokes='u')

In [ ]:
fig = plt.figure(figsize=(20,13))
fs = 16
stokes = 'u'

dragons_low = copy.deepcopy(dragons[stokes])[0]
print(dragons_low.shape)
dragons_low[825:1440] = np.nan
    
crange = SkyCoord([24,0], [-24,86], frame=ICRS, unit=(u.hr, u.deg))
plt.subplots_adjust(hspace=0.0)
    
ax   = fig.add_subplot(111, projection=wcs.celestial)

freq_title = str(np.round(freq_list[fidx]/1e6,3))+' MHz'

ax.set_title('CHIME and DRAGONS '+freq_title,fontsize=fs)

im  = ax.imshow(combined[stokes][0],vmin=-1,vmax=1,cmap='RdBu_r')
ax.imshow(dragons_low,vmin=-1,vmax=1,cmap='RdBu_r')

ax.set_ylim(wcs.world_to_pixel(crange)[1])
ax.set_xlim(wcs.world_to_pixel(crange)[0])
cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.65, pad=0.005, aspect=12)
cbar.set_label(label='K',fontsize=fs)
cbar.ax.tick_params(labelsize=fs) 
ax.set_xlabel(' ')
ax.set_ylabel('dec',fontsize=fs)
ax.tick_params(axis='both',labelsize=fs)
ax.set_xlabel('RA',fontsize=fs)

plt.savefig('../PLOTS/chime_with_dragons_u.png')

In [ ]:
dragons_low_q = copy.deepcopy(dragons['q'])[0]
dragons_low_u = copy.deepcopy(dragons['u'])[0]

combined_high_q = copy.deepcopy(combined['q'])[0]
combined_high_u = copy.deepcopy(combined['u'])[0]

dragons_low_q[825:1440] = 0
dragons_low_u[825:1440] = 0

combined_high_q[0:825] = 0
combined_high_u[0:825] = 0

full_sky_q = dragons_low_q + combined_high_q
full_sky_u = dragons_low_q + combined_high_u

full_sky_q[0:540] = np.nan
full_sky_u[0:540] = np.nan

In [ ]:
fig = plt.figure(figsize=(20,13))
fs = 16

crange = SkyCoord([24,0], [-24,86], frame=ICRS, unit=(u.hr, u.deg))
plt.subplots_adjust(hspace=0.0)
    
ax   = fig.add_subplot(111, projection=wcs.celestial)
freq_title = str(np.round(freq_list[fidx]/1e6,3))+' MHz'

im  = ax.imshow(full_sky_u,vmin=-1,vmax=1,cmap='RdBu_r')

ax.set_title('CHIME and DRAGONS '+freq_title,fontsize=fs)
ax.set_ylim(wcs.world_to_pixel(crange)[1])
ax.set_xlim(wcs.world_to_pixel(crange)[0])
#cbar = fig.colorbar(im, ax=ax, shrink=0.65, pad=0.005, aspect=12)
#cbar.set_label(label='K',fontsize=fs)
#cbar.ax.tick_params(labelsize=fs) 
ax.set_xlabel(' ')
ax.set_ylabel('dec',fontsize=fs)
ax.tick_params(axis='both',labelsize=fs)
ax.set_xlabel('RA',fontsize=fs)

#plt.savefig('../PLOTS/chime_with_dragons_u.png')

In [ ]:
full_sky_pi = np.sqrt(full_sky_q**2 + full_sky_u**2)

In [ ]:
fig = plt.figure(figsize=(20,13))
fs = 16

crange = SkyCoord([24,0], [-24,86], frame=ICRS, unit=(u.hr, u.deg))
plt.subplots_adjust(hspace=0.0)
    
ax   = fig.add_subplot(111, projection=wcs.celestial)
freq_title = str(np.round(freq_list[fidx]/1e6,3))+' MHz'

im  = ax.imshow(full_sky_pi,vmin=0.0,vmax=1.5,cmap='gray',interpolation=None)

ax.set_title('CHIME and DRAGONS '+freq_title,fontsize=fs)
ax.set_ylim(wcs.world_to_pixel(crange)[1])
ax.set_xlim(wcs.world_to_pixel(crange)[0])
#cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.65, pad=0.005, aspect=12)
#cbar.set_label(label='K',fontsize=fs)
#cbar.ax.tick_params(labelsize=fs) 
ax.set_xlabel(' ')
ax.set_ylabel('dec',fontsize=fs)
ax.tick_params(axis='both',labelsize=fs)
ax.set_xlabel('RA',fontsize=fs)


In [ ]:
full_sky_hpx, footprint = reproject_to_healpix((full_sky_pi, wcs.to_header()),'ICRS', nside=1024)

In [ ]:
hp.newvisufunc.projview(full_sky_hpx,cmap='cubehelix',min=-0.2,max=1.5,rot=(0,0),
                            xsize=2000,cbar=False,title='',unit='',
                            projection_type='mollweide')

In [ ]:
r = hp.Rotator(coord=['C', 'G'])
full_sky_gal_hpx = np.array(r.rotate_map_pixel(full_sky_hpx))

In [ ]:
full_sky_gal_hpx[np.isnan(full_sky_gal_hpx)] = 0

In [ ]:
cmap = cm.cubehelix
cmap.set_bad(color='k')

hp.newvisufunc.projview(full_sky_gal_hpx,cmap=cmap,min=0,max=2,rot=(0,0),
                            xsize=2000,cbar=False,title='',unit='',
                            projection_type='mollweide')

In [ ]:
hduQ = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_q_cube_32bit.fits')
hduU = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_q_cube_32bit.fits')

wcs2 = WCS(hduU[0].header).dropaxis(2)

print(wcs2)

pi_dragons = np.sqrt(hduQ[0].data[100]**2 + hduU[0].data[100]**2)
print(pi_dragons.shape)

In [ ]:
plt.imshow(pi_dragons)

In [ ]:
'/srv/data/chime/chime_QU_Oct2023_400_729/
'/srv/data/dragons/Mar2024_500_1000_MHz_cube/